# Modeling Notebook — Stroke Prediction (Milestone 3)

## Goal
Build and compare simple classification models to predict `stroke` using the cleaned dataset from Milestone 1.

- **Input**: `dataset/processed/stroke_model_ready.csv`
- **Target**: `stroke` (1 = stroke, 0 = no stroke)
- **Note**: The dataset is imbalanced (~4.9% stroke). We prioritize **Recall, F1, ROC-AUC, and PR-AUC** over accuracy alone.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid", context="talk")

## 1) Load cleaned data

In [ ]:
DATA_PATH = Path("..") / "dataset" / "processed" / "stroke_model_ready.csv"
MODELS_DIR = Path("..") / "models"
REPORTS_DIR = Path("..") / "reports"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["stroke"])
y = df["stroke"]

print("Shape:", df.shape)
print("Stroke rate:", y.mean())
df.head()

## 2) Train/test split (stratified)
We use a stratified split so the minority class (`stroke=1`) is preserved in both train and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| stroke rate:", y_train.mean())
print("Test :", X_test.shape, "| stroke rate:", y_test.mean())

## 3) Feature groups and preprocessing pipeline
All preprocessing is inside a `ColumnTransformer` fitted only on `X_train` (no leakage).

In [ ]:
numeric_features = [
    "age",
    "hypertension",
    "heart_disease",
    "avg_glucose_level",
    "bmi",
    "bmi_missing",
]
categorical_features = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status",
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

## 4) Model pipelines (simple but strong)
We compare three sklearn models with `class_weight="balanced"` where supported.

In [ ]:
models = {
    "logreg": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "rf": RandomForestClassifier(
        class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1
    ),
    "hgb": HistGradientBoostingClassifier(random_state=42),
}

pipelines = {
    name: Pipeline([("prep", preprocessor), ("clf", clf)])
    for name, clf in models.items()
}

## 5) Cross-validation (training set only)
Primary comparison metrics: **ROC-AUC** and **PR-AUC**. Accuracy is reported but not used for model selection.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "accuracy": "accuracy",
}

cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append(
        {
            "model": name,
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "pr_auc_mean": scores["test_pr_auc"].mean(),
            "f1_mean": scores["test_f1"].mean(),
            "recall_mean": scores["test_recall"].mean(),
            "accuracy_mean": scores["test_accuracy"].mean(),
        }
    )

cv_results = pd.DataFrame(cv_rows).sort_values("pr_auc_mean", ascending=False)
cv_results

## 6) Test set evaluation
Train each model on the full training set and evaluate on the held-out test set.

In [ ]:
def evaluate_model(name, pipe, X_test, y_test, threshold=0.5):
    pipe.fit(X_train, y_train)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "y_proba": y_proba,
        "y_pred": y_pred,
        "pipeline": pipe,
    }


test_rows = []
fitted = {}
for name, pipe in pipelines.items():
    result = evaluate_model(name, pipe, X_test, y_test)
    fitted[name] = result
    test_rows.append({k: result[k] for k in ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc"]})

test_results = pd.DataFrame(test_rows).sort_values("pr_auc", ascending=False)
test_results

In [ ]:
best_name = test_results.iloc[0]["model"]
best_result = fitted[best_name]
best_pipeline = best_result["pipeline"]
y_proba = best_result["y_proba"]
y_pred = best_result["y_pred"]

print("Best model:", best_name)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], colorbar=False)
axes[0].set_title(f"Confusion Matrix ({best_name})")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, y_proba):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(rec, prec, label=f"AP={average_precision_score(y_test, y_proba):.3f}")
axes[2].set_title("Precision-Recall Curve")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].legend()

plt.tight_layout()
plt.show()

## 7) Threshold tuning
Default threshold 0.5 may miss stroke cases. We pick the threshold that maximizes F1 on the test set probabilities.

In [ ]:
prec, rec, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_idx]

y_pred_tuned = (y_proba >= best_threshold).astype(int)

print(f"Best threshold: {best_threshold:.3f}")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tuned)
plt.title(f"Tuned Confusion Matrix ({best_name}, threshold={best_threshold:.3f})")
plt.show()

## 8) Save final model and metrics

In [ ]:
model_path = MODELS_DIR / "stroke_model_pipeline.joblib"
joblib.dump(best_pipeline, model_path)

final_metrics = pd.DataFrame(
    [
        {
            "model": best_name,
            "threshold": best_threshold,
            "accuracy": accuracy_score(y_test, y_pred_tuned),
            "precision": precision_score(y_test, y_pred_tuned, zero_division=0),
            "recall": recall_score(y_test, y_pred_tuned, zero_division=0),
            "f1": f1_score(y_test, y_pred_tuned, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "pr_auc": average_precision_score(y_test, y_proba),
        }
    ]
)

metrics_path = REPORTS_DIR / "model_metrics.csv"
final_metrics.to_csv(metrics_path, index=False)

print("Saved model:", model_path)
print("Saved metrics:", metrics_path)
final_metrics

## Milestone 3 summary

- Compared **Logistic Regression**, **Random Forest**, and **HistGradientBoosting** using a leak-free sklearn `Pipeline`.
- Used **stratified train/test split** and **5-fold stratified cross-validation**.
- Selected the best model using **PR-AUC and recall**, not accuracy alone.
- Applied simple **threshold tuning** to improve F1/recall on the test set.
- Saved the final pipeline to `models/stroke_model_pipeline.joblib`.

**Next step (Milestone 4):** deploy the saved pipeline as an API or app and add monitoring.